In [0]:
import requests

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

WEATHER_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m",
]

RAW_DIR = "/Volumes/workspace/default/products/european_climate/raw"
dbutils.fs.mkdirs(RAW_DIR)

print("Base endpoint:", BASE_URL)
print("Weather variables:", len(WEATHER_VARIABLES))


Base endpoint: https://archive-api.open-meteo.com/v1/archive
Weather variables: 12


This is a small ingestion demonstration, not the nine-city production load.  production ingestion is implemented separately in Notebooks 07 and 08.

In [0]:

CITY = {
    "city": "London",
    "country": "United Kingdom",
    "region": "Western Europe",
    "latitude": 51.5074,
    "longitude": -0.1278,
}

START_DATE = "2024-01-01"
END_DATE = "2024-01-07"

PARAMS = {
    "latitude": CITY["latitude"],
    "longitude": CITY["longitude"],
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": ",".join(WEATHER_VARIABLES),
    "timezone": "UTC",
    "models": "era5",
}

PARAMS


{'latitude': 51.5074,
 'longitude': -0.1278,
 'start_date': '2024-01-01',
 'end_date': '2024-01-07',
 'hourly': 'temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,weather_code,pressure_msl,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m',
 'timezone': 'UTC',
 'models': 'era5'}

In [0]:
def fetch_raw_weather(city, start_date, end_date, timeout=60):
    params = {
        "latitude": city["latitude"],
        "longitude": city["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(WEATHER_VARIABLES),
        "timezone": "UTC",
        "models": "era5",
    }

    response = requests.get(BASE_URL, params=params, timeout=timeout)
    response.raise_for_status()

    print("Status code:", response.status_code)
    print("Exact endpoint URL:")
    print(response.url)
    print(f"Raw response size: {len(response.content) / (1024 ** 2):.3f} MB")

    return response


In [0]:
def save_raw_response(response, city_name, start_date, end_date):
    # Save response.text directly so the file remains the exact API JSON response.
    safe_city = city_name.lower().replace(" ", "_")
    file_name = f"{safe_city}_{start_date}_{end_date}.json"
    raw_path = f"{RAW_DIR}/{file_name}"

    dbutils.fs.put(raw_path, response.text, overwrite=True)

    print("Saved raw response:", raw_path)
    return raw_path


In [0]:
response = fetch_raw_weather(
    CITY,
    START_DATE,
    END_DATE,
)

raw_path = save_raw_response(
    response,
    CITY["city"],
    START_DATE,
    END_DATE,
)

# Minimal validation without changing the raw response.
raw_data = response.json()

assert "hourly" in raw_data, "Missing 'hourly' object in API response"
assert "time" in raw_data["hourly"], "Missing hourly.time array"

hour_count = len(raw_data["hourly"]["time"])

print("Hourly observations:", hour_count)
print("Top-level keys:", list(raw_data.keys()))
print("Hourly keys:", list(raw_data["hourly"].keys()))
print("Raw file:", raw_path)


Status code: 200
Exact endpoint URL:
https://archive-api.open-meteo.com/v1/archive?latitude=51.5074&longitude=-0.1278&start_date=2024-01-01&end_date=2024-01-07&hourly=temperature_2m%2Crelative_humidity_2m%2Capparent_temperature%2Cprecipitation%2Crain%2Csnowfall%2Cweather_code%2Cpressure_msl%2Csurface_pressure%2Ccloud_cover%2Cwind_speed_10m%2Cwind_direction_10m&timezone=UTC&models=era5
Raw response size: 0.012 MB
Wrote 12890 bytes.
Saved raw response: /Volumes/workspace/default/products/european_climate/raw/london_2024-01-01_2024-01-07.json
Hourly observations: 168
Top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Hourly keys: ['time', 'temperature_2m', 'relative_humidity_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'weather_code', 'pressure_msl', 'surface_pressure', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m']
Raw file: /Volumes/workspace/default